# 04 — Add padded DINO features while preserving the exact original split


Reuse the base dataset/split snapshot and create the DINO cache if absent. This never reruns splitting or GT matching. Changes to padding/model/preprocessing refresh only the new feature cache.


In [1]:
from pathlib import Path
import json
import copy
import torch
from config import CONFIG, ROOT, fingerprint, reuse_base_caches, require_cache, run_stage
reuse_base_caches()
CACHE = CONFIG['cache_dir']
BASE = CACHE / 'base_samples.pt'
assert BASE.exists(), 'Set base_experiment in config.py to the original experiment with cache/samples.pt.'
base = torch.load(BASE, map_location='cpu', weights_only=True)
# Original feature settings are frozen with the base snapshot; do not mix changed encoders into that dataset.
provenance_file = CACHE / 'base_cache_provenance.json'
if provenance_file.exists():
    from config import model_files
    provenance = json.loads(provenance_file.read_text(encoding='utf-8'))
    if 'feature_settings' in provenance:
        for name, value in provenance['feature_settings'].items():
            assert CONFIG[name] == value, f'Base feature setting changed: {name}. Use a consistent base snapshot.'
        for name, value in provenance['model_files'].items():
            assert model_files(CONFIG[name]) == value, f'Base encoder files changed: {name}.'
require_cache(CACHE / 'sequences.json', '01_prepare_sequence_cache.ipynb')
sequences = json.loads((CACHE / 'sequences.json').read_text(encoding='utf-8'))
assert len(sequences) == len(base['samples'])
for seq, sample in zip(sequences, base['samples']):
    assert seq['sequence_id'] == sample['sequence_id']
    assert seq['boxes'] == sample['boxes'] and seq['pages'] == sample['pages'], 'Base sequence data changed; create a consistent snapshot.'
# This checks the current feature signature; a matching cache performs no encoder inference.
run_stage('03a_cache_dino_crops.ipynb')
dino = torch.load(CACHE / 'dino.pt', map_location='cpu', weights_only=True)
signature = fingerprint({'base_samples': base['signature'], 'dino': dino['signature'], 'version': 1})
SAMPLES = CACHE / 'samples.pt'
saved = torch.load(SAMPLES, map_location='cpu', weights_only=True) if SAMPLES.exists() else None
print('Reuse matching padded samples:', bool(saved and saved['signature'] == signature))


Running prerequisite: 03a_cache_dino_crops.ipynb
Local encoder: C:\MODELS\dinov2-small
Padding per side: 0.2 | matching feature cache: True
DINO features: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\dino.pt | sequences: 80
Reuse matching padded samples: False


Join one DINO vector per box and preserve train/validation indices, class weights, labels and original geometry. The only new model input is sample["dino"].


In [2]:
if saved is None or saved['signature'] != signature:
    saved = copy.deepcopy(base)
    for sample in saved['samples']:
        sid = sample['sequence_id']
        ids = [box['box_id'] for box in sample['boxes']]
        sample['dino'] = torch.stack([dino['vectors'][sid][bid] for bid in ids])
        assert sample['dino'].shape == (len(ids), 384) and torch.isfinite(sample['dino']).all()
    saved['signature'] = signature
    saved['base_samples_signature'] = base['signature']
    saved['feature_config'] = dino['feature_config']
    saved['dino_signature'] = dino['signature']
    torch.save(saved, SAMPLES)
assert saved['train_indices'] == base['train_indices'] and saved['val_indices'] == base['val_indices']
assert torch.equal(saved['class_weights'], base['class_weights'])
from training_runs import validate_split
validate_split(saved)
print('Exact original split:', len(saved['train_indices']), 'train /', len(saved['val_indices']), 'validation')
print('Original train class counts:', saved['train_counts'].tolist())
print('Feature widths: text 768, DETR 256, DINO 384, geometry 4, tail 2; fused input 688.')
print('Saved:', SAMPLES)


Exact original split: 64 train / 16 validation
Original train class counts: [1142, 50]
Feature widths: text 768, DETR 256, DINO 384, geometry 4, tail 2; fused input 688.
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\samples.pt
